# Bước 2: Tái lập Y văn gốc (Baseline Model)

**Mục tiêu:** xây dựng mô hình phân loại cơ bản và đối chiếu với bài báo gốc
`chicco2020machine` — Chicco & Jurman (2020), *BMC Medical Informatics and
Decision Making* — Random Forest, Accuracy 74,0%, MCC 0,384 (12 đặc trưng,
không dùng `time`).

In [1]:
import sys
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, matthews_corrcoef, f1_score, classification_report

def _load_heart_failure_data():
    """Tải dữ liệu cục bộ (../data/...) nếu có (chạy trong repo HMYT đã
    clone); nếu không (mở độc lập qua Colab/Kaggle, không có thư mục data/
    đi kèm) tự động tải từ mirror công khai trên hmyt-book (repo Public,
    xác minh 23/09/2026)."""
    import os
    local_path = '../data/heart_failure_clinical_records_dataset.csv'
    remote_url = ('https://raw.githubusercontent.com/fossbk-spec/hmyt-book/gh-pages/'
                  'labs_chuyen_de/ch02_suy_tim_risk_dxai/data/'
                  'heart_failure_clinical_records_dataset.csv')
    path = local_path if os.path.exists(local_path) else remote_url
    if path == remote_url:
        print(f"[i] Không tìm thấy dữ liệu cục bộ — tự động tải từ mirror công khai:\n    {remote_url}")
    return pd.read_csv(path).rename(columns={'death_event': 'DEATH_EVENT'})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = _load_heart_failure_data()
FEATURE_COLS = [c for c in df.columns if c != 'DEATH_EVENT']
X, y = df[FEATURE_COLS], df['DEATH_EVENT'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}")

Train: 239  Test: 60


## 1. Logistic Regression (hệ quy chiếu tuyến tính)

In [2]:
logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logreg.fit(X_train_s, y_train)
y_pred_lr = logreg.predict(X_test_s)

acc_lr = accuracy_score(y_test, y_pred_lr)
mcc_lr = matthews_corrcoef(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)
print(f"Logistic Regression -> Accuracy: {acc_lr:.4f} | MCC: {mcc_lr:.4f} | F1: {f1_lr:.4f}")

Logistic Regression -> Accuracy: 0.8167 | MCC: 0.5563 | F1: 0.6667


## 2. Random Forest (đúng phương pháp nổi bật trong bài báo gốc)

In [3]:
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf.fit(X_train_s, y_train)
y_pred_rf = rf.predict(X_test_s)

acc_rf = accuracy_score(y_test, y_pred_rf)
mcc_rf = matthews_corrcoef(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
print(f"Random Forest -> Accuracy: {acc_rf:.4f} | MCC: {mcc_rf:.4f} | F1: {f1_rf:.4f}")
print()
print(classification_report(y_test, y_pred_rf, target_names=['Sống sót', 'Tử vong']))

Random Forest -> Accuracy: 0.8167 | MCC: 0.5563 | F1: 0.6667

              precision    recall  f1-score   support

    Sống sót       0.83      0.93      0.87        41
     Tử vong       0.79      0.58      0.67        19

    accuracy                           0.82        60
   macro avg       0.81      0.75      0.77        60
weighted avg       0.81      0.82      0.81        60



## 3. Đối chiếu Y văn — `chicco2020machine`

| Mô hình | Accuracy (bài báo gốc) | MCC (bài báo gốc) | Accuracy (notebook này) | MCC (notebook này) |
|---|:---:|:---:|:---:|:---:|
| Random Forest | 74,0% | 0,384 | *(xem output cell dưới — số liệu THẬT từ lần chạy này, không gán cứng)* | *(idem)* |

> ⚠️ Lưu ý phương pháp luận: bài báo gốc dùng 10-fold cross-validation lặp
> lại 100 lần và báo cáo trung bình; notebook này (đúng theo yêu cầu Bước 1)
> chỉ dùng 1 lần Train/Test split 80/20 cố định `random_state=42` — vì vậy
> kết quả 1 lần chạy có thể dao động quanh mốc 74%/0,384 chứ không nhất
> thiết trùng khớp tuyệt đối. Đây là hạn chế cần nêu rõ trong báo cáo
> Bước 5, không phải lỗi của pipeline.

In [4]:
print("=== BẢNG SO SÁNH VỚI Y VĂN (số liệu thật từ lần chạy này) ===")
print(f"{'Mô hình':<22}{'Accuracy':>12}{'MCC':>10}{'F1':>10}")
print(f"{'Logistic Regression':<22}{acc_lr:>12.4f}{mcc_lr:>10.4f}{f1_lr:>10.4f}")
print(f"{'Random Forest':<22}{acc_rf:>12.4f}{mcc_rf:>10.4f}{f1_rf:>10.4f}")
print(f"{'chicco2020machine (RF, y văn)':<22}{'0.7400':>12}{'0.3840':>10}{'-':>10}")
delta_acc = acc_rf - 0.740
delta_mcc = mcc_rf - 0.384
print(f"\nChênh lệch so với y văn: Accuracy {delta_acc:+.4f} | MCC {delta_mcc:+.4f}")

=== BẢNG SO SÁNH VỚI Y VĂN (số liệu thật từ lần chạy này) ===
Mô hình                   Accuracy       MCC        F1
Logistic Regression         0.8167    0.5563    0.6667
Random Forest               0.8167    0.5563    0.6667
chicco2020machine (RF, y văn)      0.7400    0.3840         -

Chênh lệch so với y văn: Accuracy +0.0767 | MCC +0.1723


## 4. Tổng kết Bước 2

Kết quả Random Forest trên tập Test giữ lại (n=60, 20% của 299) được ghi
nhận trung thực ở cell trên — không làm tròn/điều chỉnh để khớp y văn.
Sai khác (nếu có) với mốc 74,0%/0,384 của `chicco2020machine` đến từ khác
biệt phương pháp đánh giá (1-split vs. 100×10-fold CV) và cỡ mẫu Test nhỏ
(n=60) khiến phương sai ước lượng cao — đúng hạn chế đã nêu ở Mục 2.10 của
Chương 2 ("cỡ mẫu nhỏ, thiếu validation bên ngoài").

**Tiếp theo:** [`3_improvement.ipynb`](./3_improvement.ipynb) — Bước 3, áp
dụng SMOTE hướng tới mốc 92,6% của `ishaq2021improving`.